# Assignment 3: Time Series Prediction of Sepsis

Authors:  
Max Masuch  
Ismail Mohammed

## Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pandas import read_csv
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout, Input
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, multilabel_confusion_matrix, accuracy_score, roc_auc_score

## Load data

This section should load the raw dataset for the task.  
Remember to use relative paths to load any files in the notebook.

In [138]:
df_a = pd.read_csv('raw_data/sepsisexp_timeseries_partition-A.tsv', sep='\t')

df_b = pd.read_csv('raw_data/sepsisexp_timeseries_partition-B.tsv', sep='\t')

df_c = pd.read_csv('raw_data/sepsisexp_timeseries_partition-C.tsv', sep='\t')

df_d = pd.read_csv('raw_data/sepsisexp_timeseries_partition-D.tsv', sep='\t')

print(df_a.head())

      id  sepsis  severity  timestep  respiratory_minute_volume  heart_rate  \
0  12292       0       0.0       0.0                   0.190898    0.424464   
1  12292       0       0.0       0.5                   0.157654    0.667394   
2  12292       0       0.0       1.0                   0.024678    0.618808   
3  12292       0       0.0       1.5                  -0.208030    0.278706   
4  12292       0       0.0       2.0                  -0.108298   -0.352912   

   leukocytes  temperature  partial_co2  respiratory_rate  ...   calcium  \
0    0.301015    -0.168117    -0.275272          1.879692  ...  1.019083   
1    0.301015    -0.168117    -0.275272          1.708485  ...  1.019083   
2    0.301015    -0.732387     1.003408          2.050899  ... -0.868157   
3    0.301015    -0.732387     1.003408          1.366071  ... -0.868157   
4    0.301015    -0.732387     1.094023          1.537278  ... -0.868157   

   potassium  mixed_venous_oxygen_saturation  urine_output  net bala

In [139]:
train_df = pd.concat([df_a, df_b], axis=0).reset_index(drop=True)
val_df = df_c.reset_index(drop=True)
test_df = df_d.reset_index(drop=True)

In [140]:
excluded = ['id', 'sepsis', 'severity', 'timestep']
features = [col for col in train_df.columns if col not in excluded]

In [141]:
def add_targets(df):
    df['target_2h'] = df.groupby('id')['sepsis'].shift(-4)
    df['target_4h'] = df.groupby('id')['sepsis'].shift(-8)
    df['target_6h'] = df.groupby('id')['sepsis'].shift(-12)
    return df

In [142]:
df_train = add_targets(train_df).ffill().fillna(0)
df_val = add_targets(val_df).ffill().fillna(0)
df_test = add_targets(test_df).ffill().fillna(0)

In [143]:
def create_sequences(df, feature_cols, target_col, window_size):
    X, y = [], []
    for _, group in df.groupby('id'):
        if len(group) <= window_size:
            continue
        
        data = group[feature_cols].values
        target = group[target_col].values
        
        for i in range(len(group) - window_size):
            X.append(data[i : i + window_size])
            y.append(target[i + window_size])
            
    return np.array(X), np.array(y)

In [163]:
X_train_2h, y_train_2h = create_sequences(df_train, features, 'target_2h', 12)
X_val_2h, y_val_2h = create_sequences(df_val, features, 'target_2h', 12)
X_test_2h, y_test_2h = create_sequences(df_test, features, 'target_2h', 12)

X_train_4h, y_train_4h = create_sequences(df_train, features, 'target_4h', 24)
X_val_4h, y_val_4h = create_sequences(df_val, features, 'target_4h', 24)
X_test_4h, y_test_4h = create_sequences(df_test, features, 'target_4h', 24)

X_train_6h, y_train_6h = create_sequences(df_train, features, 'target_6h', 36)
X_val_6h, y_val_6h = create_sequences(df_val, features, 'target_6h', 36)
X_test_6h, y_test_6h = create_sequences(df_test, features, 'target_6h', 36)

## DENNA MODEL ÄR HEMSK MAX. IDK WHAT TO DO

In [168]:
def train_model(X_train, y_train, X_val, y_val):
    
    model = Sequential()

    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))

    model.add(LSTM(64, return_sequences=False))

    model.add(Dropout(0.2))

    model.add(Dense(1, activation='sigmoid'))

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.00001)

    model.compile(optimizer=optimizer, 
        loss='binary_crossentropy', 
        metrics=['accuracy', 'auc'])
    
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3, 
        restore_best_weights=True)


    model.fit(
        X_train, y_train, 
        validation_data=(X_val, y_val), 
        epochs=25, batch_size=20, 
        callbacks=[early_stop], verbose=1)
    
    return model

In [169]:
print('2h Model')
model_2h = train_model(X_train_2h, y_train_2h, X_val_2h, y_val_2h)

2h Model
Epoch 1/25
15061/15061 ━━━━━━━━━━━━━━━━━━━━ 52s 3ms/step - accuracy: 0.7594 - auc: 0.8329 - loss: 0.4916 - val_accuracy: 0.7277 - val_auc: 0.7595 - val_loss: 0.5936
Epoch 2/25
15061/15061 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - accuracy: 0.8389 - auc: 0.9151 - loss: 0.3611 - val_accuracy: 0.7091 - val_auc: 0.7591 - val_loss: 0.6437
Epoch 3/25
15061/15061 ━━━━━━━━━━━━━━━━━━━━ 49s 3ms/step - accuracy: 0.8769 - auc: 0.9474 - loss: 0.2864 - val_accuracy: 0.7027 - val_auc: 0.7532 - val_loss: 0.7129
Epoch 4/25
15061/15061 ━━━━━━━━━━━━━━━━━━━━ 49s 3ms/step - accuracy: 0.8983 - auc: 0.9627 - loss: 0.2414 - val_accuracy: 0.6950 - val_auc: 0.7450 - val_loss: 0.7952


In [170]:
print('4h Model')
model_4h = train_model(X_train_4h, y_train_4h, X_val_4h, y_val_4h)

4h Model
Epoch 1/25
14678/14678 ━━━━━━━━━━━━━━━━━━━━ 83s 6ms/step - accuracy: 0.7638 - auc: 0.8374 - loss: 0.4824 - val_accuracy: 0.7175 - val_auc: 0.7454 - val_loss: 0.6147
Epoch 2/25
14678/14678 ━━━━━━━━━━━━━━━━━━━━ 74s 5ms/step - accuracy: 0.8616 - auc: 0.9312 - loss: 0.3250 - val_accuracy: 0.7149 - val_auc: 0.7421 - val_loss: 0.6973
Epoch 3/25
14678/14678 ━━━━━━━━━━━━━━━━━━━━ 74s 5ms/step - accuracy: 0.9041 - auc: 0.9635 - loss: 0.2387 - val_accuracy: 0.7067 - val_auc: 0.7293 - val_loss: 0.8185
Epoch 4/25
14678/14678 ━━━━━━━━━━━━━━━━━━━━ 73s 5ms/step - accuracy: 0.9283 - auc: 0.9773 - loss: 0.1874 - val_accuracy: 0.6977 - val_auc: 0.7225 - val_loss: 0.9297


In [171]:
print('6h Model')
model_6h = train_model(X_train_6h, y_train_6h, X_val_6h, y_val_6h)

6h Model
Epoch 1/25
14298/14298 ━━━━━━━━━━━━━━━━━━━━ 94s 7ms/step - accuracy: 0.7790 - auc: 0.8489 - loss: 0.4645 - val_accuracy: 0.7183 - val_auc: 0.7317 - val_loss: 0.6403
Epoch 2/25
14298/14298 ━━━━━━━━━━━━━━━━━━━━ 88s 6ms/step - accuracy: 0.8767 - auc: 0.9407 - loss: 0.3002 - val_accuracy: 0.7020 - val_auc: 0.7239 - val_loss: 0.7588
Epoch 3/25
14298/14298 ━━━━━━━━━━━━━━━━━━━━ 87s 6ms/step - accuracy: 0.9152 - auc: 0.9683 - loss: 0.2187 - val_accuracy: 0.6893 - val_auc: 0.7106 - val_loss: 0.8853
Epoch 4/25
14298/14298 ━━━━━━━━━━━━━━━━━━━━ 87s 6ms/step - accuracy: 0.9385 - auc: 0.9806 - loss: 0.1679 - val_accuracy: 0.6870 - val_auc: 0.7082 - val_loss: 0.9907


In [172]:
X_test_2h, y_test_2h = create_sequences(df_test, features, 'target_2h', 12)
X_test_4h, y_test_4h = create_sequences(df_test, features, 'target_4h', 24)
X_test_6h, y_test_6h = create_sequences(df_test, features, 'target_6h', 36)

In [173]:
results_2h = model_2h.evaluate(X_test_2h, y_test_2h, verbose=1)
results_4h = model_4h.evaluate(X_test_4h, y_test_4h, verbose=1)
results_6h = model_6h.evaluate(X_test_6h, y_test_6h, verbose=1)

4178/4178 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.7336 - auc: 0.7888 - loss: 0.5448
4059/4059 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.7433 - auc: 0.7934 - loss: 0.5401
3940/3940 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.7552 - auc: 0.8111 - loss: 0.5169


In [167]:
def get_feature_importance(model, X, y, feature_names):
    baseline_auc = roc_auc_score(y, model.predict(X, verbose=0))
    importances = []
    
    for i in range(len(feature_names)):
        save_col = X[:, :, i].copy()
        X[:, :, i] = np.random.permutation(X[:, :, i])
        
        permuted_auc = roc_auc_score(y, model.predict(X, verbose=0))
        importances.append(baseline_auc - permuted_auc)
        
        X[:, :, i] = save_col
        
    return pd.Series(importances, index=feature_names).sort_values(ascending=False)

importance_2h = get_feature_importance(model_2h, X_test_2h, y_test_2h, features)
print("\nTopp 10 viktigaste särdrag för 2h:")
print(importance_2h.head(10))


Topp 10 viktigaste särdrag för 2h:
base_excess                  0.046074
hemoglobin                   0.018448
c-reactive_protein           0.016350
partial_co2                  0.015313
stroke_volume                0.013835
heart_time_volume            0.012670
respiratory_minute_volume    0.011900
creatinine                   0.009031
respiratory_rate             0.008689
fraction_of_inspired_o2      0.008670
dtype: float64


## Task 1: This is the title of task 1

This section should contain the solution of task 1.

It is mandatory to maintain the headings for each task.  
OPTIONALLY, you can use one level down (###) to organize subsessions of the assignments.

Use markdown cells like this one to include:
- Discussion points.
- References to specific sources of code that you might have used to solve the assignment.
- General commentas and explanations about your solution.

In [ ]:
# Always use comments in the code to document specific steps

## Task 2: This is the title of task 2

This section should contain the solution of task 2.

## Results and Discussion

This section should contain:
- Results.
- Summary of best model performance:
    - Name of best model file as saved in /models.
    - Relevant scores such as: accuracy, precision, recall, F1-score, etc.
- Key discussion points.

In [ ]:
# Always use comments in the code to document specific steps